# FFCA-Pruned MLP Retraining

Retrains all 20 original experiments (5 models × 4 lead times) using only the features
that the Feature Function Curvatura Analysis (FFCA) rated as **CONFIDENTLY KEEP**,
**KEEP (stable)**, or **MONITOR (borderline)** across the 30-member ensemble.

Everything else is identical to the original training run:
- Same hyperparameters (layers, neurons, learning rate) — extracted from saved `.h5` files
- Same training protocol (30 ensemble members, different seeds, early stopping, batch size)
- Same train / val / test year split (2017-2022 train, 2023 val, 2024 test)
- Same loss function (`mean_squared_error`)
- Same y-bounds (full-dataset min/max with 20% extrapolation buffer)

**Results land in** `results/<experiment_name>_ffca/MLP/` — ready to compare against
the originals in `results/<experiment_name>/MLP/`.

Run cells 1–3 once to set up, then run **Cell 4** to train all 20 experiments.

In [2]:
# ─── Cell 1: Environment ──────────────────────────────────────────────────────
import os, sys, json, re, time, warnings
from collections import defaultdict
from pathlib import Path

warnings.filterwarnings('ignore')

# Resolve repo root whether notebook is launched from notebooks/ or repo root
REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
os.chdir(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))

# ── GPU / CPU selection ───────────────────────────────────────────────────────
# Set USE_GPU = False to force CPU-only (useful for debugging on login nodes)
USE_GPU = True
if not USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # suppress TF info/warning spam

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Lambda
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
import sklearn.metrics as skm

print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
print(f'GPUs       : {len(tf.config.list_physical_devices("GPU"))} device(s) visible')
print(f'Working dir: {os.getcwd()}')

TensorFlow : 2.18.1
Keras      : 3.11.2
GPUs       : 0 device(s) visible
Working dir: C:\Users\jcham070\ml-miami-compound-flood-predictions


In [3]:
# ─── Cell 2: Configuration ────────────────────────────────────────────────────

# FFCA decision categories to include in the pruned feature set
KEEP_DECISIONS = {'CONFIDENTLY KEEP', 'KEEP (stable)', 'MONITOR (borderline)'}

# Training constants — must match original pipeline exactly
N_ENSEMBLE  = 30        # ensemble members per experiment
MAX_EPOCHS  = 10_000    # early stopping will cut this short
BATCH_SIZE  = 64
PATIENCE    = 20        # early stopping patience (epochs without improvement)
Y_BUFFER    = 0.20      # 20% extrapolation buffer on y_max
VAL_YEAR    = 2023      # held-out year used only for early stopping in ensemble training

RESULTS_ROOT = 'results'

# ─── Experiment registry ──────────────────────────────────────────────────────
# hp values are extracted directly from the saved original .h5 model files
# (1 hidden layer won every grid search; LR was 1e-3 in 19/20 cases)

EXPERIMENTS = [
    # ── Model 1: Measurements Only ───────────────────────────────────────────
    dict(
        name      = '3hr_measured_sigmoid',
        ffca      = 'FFCA_results/Measurements Only/3hr_measured_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 3,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=100, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '6hr_measured_sigmoid',
        ffca      = 'FFCA_results/Measurements Only/6hr_measured_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 6,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '12hr_measured_sigmoid',
        ffca      = 'FFCA_results/Measurements Only/12hr_measured_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 12,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=100, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '24hr_measured_sigmoid',
        ffca      = 'FFCA_results/Measurements Only/24hr_measured_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 24,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),

    # ── Model 2: Predicted Ocean Water Levels ────────────────────────────────
    dict(
        name      = '3hr_perfect_prog_wls_sigmoid',
        ffca      = 'FFCA_results/Predicted Ocean Water Levels/3hr_perfect_prog_wls_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 3,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '6hr_perfect_prog_wls_sigmoid',
        ffca      = 'FFCA_results/Predicted Ocean Water Levels/6hr_perfect_prog_wls_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 6,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-4, activation='relu'),  # NOTE: 1e-4, only exception
    ),
    dict(
        name      = '12hr_perfect_prog_wls_sigmoid',
        ffca      = 'FFCA_results/Predicted Ocean Water Levels/12hr_perfect_prog_wls_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 12,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=100, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '24hr_perfect_prog_wls_sigmoid',
        ffca      = 'FFCA_results/Predicted Ocean Water Levels/24hr_perfect_prog_wls_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 24,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=100, lr=1e-3, activation='relu'),
    ),

    # ── Model 3: Predicted Rainfall ──────────────────────────────────────────
    dict(
        name      = '3hr_perfect_prog_rain_sigmoid',
        ffca      = 'FFCA_results/Predicted Rainfall/3hr_perfect_prog_rain_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 3,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '6hr_perfect_prog_rain_sigmoid',
        ffca      = 'FFCA_results/Predicted Rainfall/6hr_perfect_prog_rain_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 6,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '12hr_perfect_prog_rain_sigmoid',
        ffca      = 'FFCA_results/Predicted Rainfall/12hr_perfect_prog_rain_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 12,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=100, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '24hr_perfect_prog_rain_sigmoid',
        ffca      = 'FFCA_results/Predicted Rainfall/24hr_perfect_prog_rain_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 24,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),

    # ── Model 4: Predicted Gate Opening ──────────────────────────────────────
    dict(
        name      = '3hr_perfect_prog_gate_sigmoid',
        ffca      = 'FFCA_results/Predicted Gate Opening/3hr_perfect_prog_gate_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 3,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '6hr_perfect_prog_gate_sigmoid',
        ffca      = 'FFCA_results/Predicted Gate Opening/6hr_perfect_prog_gate_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 6,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '12hr_perfect_prog_gate_sigmoid',
        ffca      = 'FFCA_results/Predicted Gate Opening/12hr_perfect_prog_gate_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 12,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '24hr_perfect_prog_gate_sigmoid',
        ffca      = 'FFCA_results/Predicted Gate Opening/24hr_perfect_prog_gate_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 24,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=100, lr=1e-3, activation='relu'),
    ),

    # ── Model 5: Predictions All Inputs ──────────────────────────────────────
    dict(
        name      = '3hr_perfect_prog_all_inputs_sigmoid',
        ffca      = 'FFCA_results/Predictions All Inputs/3hr_perfect_prog_all_inputs_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 3,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '6hr_perfect_prog_all_inputs_sigmoid',
        ffca      = 'FFCA_results/Predictions All Inputs/6hr_perfect_prog_all_inputs_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 6,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=100, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '12hr_perfect_prog_all_inputs_sigmoid',
        ffca      = 'FFCA_results/Predictions All Inputs/12hr_perfect_prog_all_inputs_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 12,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),
    dict(
        name      = '24hr_perfect_prog_all_inputs_sigmoid',
        ffca      = 'FFCA_results/Predictions All Inputs/24hr_perfect_prog_all_inputs_sigmoid/report.json',
        data      = 'data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv',
        lead_time = 24,  target = 'gwl',
        train_years = list(range(2017, 2024)),  test_years = [2024],
        hp = dict(num_layers=1, neurons=200, lr=1e-3, activation='relu'),
    ),
]

print(f'Registered {len(EXPERIMENTS)} experiments.')

Registered 20 experiments.


In [4]:
# ─── Cell 3: Helper Functions ─────────────────────────────────────────────────

# ── FFCA parsing ─────────────────────────────────────────────────────────────

def parse_feature(feat_name):
    """'gwl_t-5' -> ('gwl', -5),  'rain_t+3' -> ('rain', 3),  'gwl' -> ('gwl', 0)"""
    m = re.match(r'^(.+?)_t([+-]?\d+)$', feat_name)
    return (m.group(1), int(m.group(2))) if m else (feat_name, 0)


def load_ffca_selected(report_path, keep_decisions=KEEP_DECISIONS):
    """
    Reads a FFCA report.json and returns the features whose trust decision
    falls within keep_decisions.

    Returns:
        dict mapping variable prefix -> sorted list of lag integers
        e.g. {'gwl': [-9, -5, -1, 0], 'rain': [-22, -20, ..., 0, 1, 2, 3]}
    """
    with open(report_path) as f:
        data = json.load(f)
    selected = defaultdict(set)
    for feat, info in data['trust'].items():
        if info['decision'] in keep_decisions:
            prefix, lag = parse_feature(feat)
            selected[prefix].add(lag)
    return {p: sorted(lags) for p, lags in selected.items()}


# ── Feature matrix construction ───────────────────────────────────────────────

def build_feature_matrix(df_raw, selected_features, target_col, lead_time):
    """
    Builds the ML feature matrix using ONLY the FFCA-selected lags.
    Handles non-contiguous lag lists — no assumption of a contiguous window.

    Shift convention (matches original pipeline create_lagged_columns):
        lag < 0  (past)  : df[col].shift(-lag)  e.g. lag=-5 -> shift(5)  -> past
        lag > 0  (future): df[col].shift(-lag)  e.g. lag=+3 -> shift(-3) -> future
        lag = 0           : original column, already present in df

    Returns:
        df        : DataFrame with feature columns + target column, NaNs dropped
        feat_cols : ordered list of feature column names
        target    : target column name (e.g. 'gwl_t+24')
    """
    df = df_raw.copy()
    feat_cols = []

    # Sort prefixes for reproducible column ordering
    for prefix in sorted(selected_features.keys()):
        if prefix not in df.columns:
            print(f'  WARNING: "{prefix}" not found in dataframe — skipping')
            continue
        for lag in sorted(selected_features[prefix]):
            if lag == 0:
                col = prefix                        # current value; already exists
            elif lag < 0:
                col = f'{prefix}_t{lag}'            # e.g. gwl_t-5
                df[col] = df[prefix].shift(-lag)    # shift(5) -> 5 hrs in the past
            else:
                col = f'{prefix}_t+{lag}'           # e.g. rain_t+3
                df[col] = df[prefix].shift(-lag)    # shift(-3) -> 3 hrs in the future
            feat_cols.append(col)

    # Create target column
    target = f'{target_col}_t+{lead_time}'
    df[target] = df[target_col].shift(-lead_time)

    df = df[feat_cols + [target]].dropna()
    return df, feat_cols, target


def split_by_years(df, years):
    """Returns (rows IN years, rows NOT IN years)."""
    mask = df.index.year.isin(years)
    return df[mask].copy(), df[~mask].copy()


# ── Evaluation metrics (same as src/evaluation/metrics.py) ───────────────────

def cf_pct(y_true, y_pred, cm):
    """Central frequency: % of predictions within ±cm of the true value."""
    return float(np.mean(np.abs(y_true - y_pred) <= cm / 100) * 100)


def evaluate(y_true, y_pred):
    return {
        'CF_15CM': cf_pct(y_true, y_pred, 15),
        'CF_5CM':  cf_pct(y_true, y_pred, 5),
        'CF_1CM':  cf_pct(y_true, y_pred, 1),
        'RMSE':    float(skm.root_mean_squared_error(y_true, y_pred)),
        'MAE':     float(skm.mean_absolute_error(y_true, y_pred)),
        'MEDAE':   float(skm.median_absolute_error(y_true, y_pred)),
        'MAPE':    float(skm.mean_absolute_percentage_error(y_true, y_pred)),
        'R2':      float(skm.r2_score(y_true, y_pred)),
    }


# ── Model construction ────────────────────────────────────────────────────────

def build_model(hp, y_min, y_max_buffered):
    """
    Builds a Keras MLP that exactly matches the original architecture:
        Dense(neurons, relu, HeNormal) -> Dropout(0.4)  [repeated num_layers times]
        -> Dense(1, sigmoid)
        -> Lambda(denormalise from [0,1] back to GWL metres)

    y_max_buffered must already include the 20% extrapolation buffer.
    """
    y_lo = float(y_min)
    y_hi = float(y_max_buffered)

    model = Sequential()
    for _ in range(hp['num_layers']):
        model.add(Dense(hp['neurons'],
                        kernel_initializer='he_normal',
                        activation=hp['activation']))
        model.add(Dropout(0.4))
    model.add(Dense(1, activation='sigmoid'))
    # Denormalisation: sigmoid output in [0,1] -> GWL in [y_lo, y_hi]
    model.add(Lambda(lambda x: x * (y_hi - y_lo) + y_lo))

    model.compile(
        loss='mean_squared_error',
        optimizer=Adam(learning_rate=hp['lr']),
    )
    return model


# ── Ensemble training ─────────────────────────────────────────────────────────

def train_ensemble(X_train, y_train, X_val, y_val,
                   hp, y_min, y_max_buffered, out_dir,
                   n=N_ENSEMBLE, epochs=MAX_EPOCHS,
                   batch=BATCH_SIZE, patience=PATIENCE):
    """
    Trains (or reloads) n ensemble members.

    Each member uses a different tf.random.set_seed(i) for weight-init
    diversity — identical to the original pipeline.

    Safe to re-run: any .h5 that already exists is loaded instead of
    retrained, so an interrupted HPC job can resume cleanly.

    Returns:
        models       : list of n trained Keras models
        stop_epochs  : list of epoch at which early stopping fired (or 'loaded')
    """
    os.makedirs(out_dir, exist_ok=True)
    models, stop_epochs = [], []

    for i in range(n):
        path = os.path.join(out_dir, f'hypermodel{i + 1}.h5')

        if os.path.exists(path):
            models.append(keras.models.load_model(path))
            stop_epochs.append('loaded')
            continue

        tf.random.set_seed(i)   # same seed strategy as original
        model = build_model(hp, y_min, y_max_buffered)

        es = EarlyStopping(
            monitor='val_loss', patience=patience, mode='min',
            restore_best_weights=True, verbose=0,
        )
        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch,
            validation_batch_size=batch,
            callbacks=[es],
            verbose=0,
        )
        model.save(path)
        models.append(model)
        # stopped_epoch == 0 means early stopping never fired (ran full epochs)
        stop_epochs.append(es.stopped_epoch if es.stopped_epoch > 0 else epochs)

    return models, stop_epochs


# ── Full single-experiment pipeline ──────────────────────────────────────────

def run_experiment(cfg):
    """
    Runs the full FFCA-pruned training pipeline for one experiment config.

    Output directory: results/<name>_ffca/MLP/
        models/hypermodel1.h5 ... hypermodel30.h5
        test/results.csv      (metrics)
        test/predictions.csv  (ensemble median + all 30 member preds + labels)

    Returns a single-row DataFrame with test metrics.
    """
    exp_name   = cfg['name'] + '_ffca'
    out_root   = os.path.join(RESULTS_ROOT, exp_name, 'MLP')
    models_dir = os.path.join(out_root, 'models')
    test_dir   = os.path.join(out_root, 'test')
    os.makedirs(test_dir, exist_ok=True)

    results_path = os.path.join(test_dir, 'results.csv')
    preds_path   = os.path.join(test_dir, 'predictions.csv')

    # Resume: skip if already done
    if os.path.exists(results_path):
        print(f'  [skip] {exp_name} — results.csv already exists')
        return pd.read_csv(results_path, index_col=0)

    t0 = time.time()

    # 1. Load raw data
    df_raw = pd.read_csv(cfg['data'], index_col=0, parse_dates=True)

    # 2. Load FFCA feature selection
    selected = load_ffca_selected(cfg['ffca'])
    n_feats  = sum(len(v) for v in selected.values())
    feat_summary = {p: f'{min(v)}..{max(v)} ({len(v)})' for p, v in selected.items()}
    print(f'  Selected {n_feats} features: {feat_summary}')

    # 3. Build feature matrix (non-contiguous lags)
    df, feat_cols, target = build_feature_matrix(
        df_raw, selected, cfg['target'], cfg['lead_time']
    )

    # 4. y bounds: computed from full raw dataset, same as original pipeline
    y_min         = float(df_raw[cfg['target']].min())
    y_max_buffered = float(df_raw[cfg['target']].max() * (1 + Y_BUFFER))

    # 5. Train / val / test split
    #    val_year  = VAL_YEAR (2023) — for ensemble early stopping only
    #    test_year = 2024          — never touched during training
    df_test, df_trainval = split_by_years(df, cfg['test_years'])
    df_val,  df_train    = split_by_years(df_trainval, [VAL_YEAR])

    X_train = df_train[feat_cols].to_numpy()
    y_train = df_train[target].to_numpy()
    X_val   = df_val[feat_cols].to_numpy()
    y_val   = df_val[target].to_numpy()
    X_test  = df_test[feat_cols].to_numpy()
    y_test  = df_test[target].to_numpy()

    print(f'  Shapes — train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}')
    print(f'  y bounds: [{y_min:.4f}, {y_max_buffered:.4f}] (buffered max)')

    # 6. Train 30-member ensemble (fixed hyperparams, no grid search)
    models, stop_epochs = train_ensemble(
        X_train, y_train, X_val, y_val,
        cfg['hp'], y_min, y_max_buffered,
        models_dir,
    )

    # 7. Ensemble predictions on test set (median across 30 members)
    all_preds    = np.array([m.predict(X_test, verbose=0).flatten() for m in models])
    median_preds = np.median(all_preds, axis=0)

    # 8. Compute metrics
    metrics = evaluate(y_test, median_preds)
    df_metrics = pd.DataFrame([metrics])
    df_metrics['model']      = 'MLP'
    df_metrics['n_features'] = len(feat_cols)
    df_metrics['experiment'] = exp_name
    df_metrics.to_csv(results_path)

    # 9. Save predictions (same format as original pipeline)
    pred_dict = {'predictions': median_preds, 'labels': y_test}
    for i, p in enumerate(all_preds, start=1):
        pred_dict[f'model_{i}'] = p
    pd.DataFrame(pred_dict, index=df_test.index).to_csv(preds_path)

    elapsed = (time.time() - t0) / 60
    n_loaded = sum(1 for e in stop_epochs if e == 'loaded')
    n_trained = N_ENSEMBLE - n_loaded
    print(
        f'  Done in {elapsed:.1f} min | '
        f'RMSE={metrics["RMSE"]*100:.2f} cm  R²={metrics["R2"]:.3f}  '
        f'CF(1cm)={metrics["CF_1CM"]:.1f}%'
    )
    if n_trained > 0:
        trained_epochs = [e for e in stop_epochs if e != 'loaded']
        print(f'  Stop epochs (trained): {trained_epochs}')

    return df_metrics


print('Functions defined.')

Functions defined.


In [5]:
# ─── Cell 4 (optional): Preview FFCA feature selection ───────────────────────
# Run this to verify what features will be used before committing to training.

print(f'{"="*72}')
print('FFCA FEATURE SELECTION PREVIEW')
print(f'Keeping decisions: {KEEP_DECISIONS}')
print(f'{"="*72}')

for cfg in EXPERIMENTS:
    selected = load_ffca_selected(cfg['ffca'])
    n_total  = sum(len(v) for v in selected.values())
    print(f'\n{cfg["name"]}  (lead={cfg["lead_time"]}h, {n_total} features, '
          f'neurons={cfg["hp"]["neurons"]}, lr={cfg["hp"]["lr"]})')
    for prefix, lags in sorted(selected.items()):
        lag_str = ', '.join(str(l) for l in lags)
        print(f'  {prefix:<12s}: [{lag_str}]  ({len(lags)} lags)')

FFCA FEATURE SELECTION PREVIEW
Keeping decisions: {'CONFIDENTLY KEEP', 'KEEP (stable)', 'MONITOR (borderline)'}

3hr_measured_sigmoid  (lead=3h, 66 features, neurons=100, lr=0.001)
  gate1       : [-24, -23, -21, 0]  (4 lags)
  gate2       : [-24, -22, -16, -15, -8]  (5 lags)
  gwl         : [-22, -17, -16, -15, -14, -13, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0]  (18 lags)
  rain        : [-18, -15, -11, -6, -5, -4, -3, -2, -1, 0]  (10 lags)
  stgH        : [-20, -19, -12, -10, -8, -6, -3, -2, -1, 0]  (10 lags)
  stgT        : [-24, -22, -21, -20, -16, -12, -11, -10, -8, -5, -3, -2, 0]  (13 lags)
  wl          : [-22, -19, -13, -3, -2, 0]  (6 lags)

6hr_measured_sigmoid  (lead=6h, 60 features, neurons=200, lr=0.001)
  gate2       : [-4, -3]  (2 lags)
  gwl         : [-22, -20, -19, -18, -14, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0]  (15 lags)
  rain        : [-24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0]  (25 lag

In [6]:
# ─── Cell 5: RUN ALL 20 EXPERIMENTS ──────────────────────────────────────────
#
# This cell trains all 20 FFCA-pruned MLP ensembles.
# Results are written to:  results/<experiment_name>_ffca/MLP/
#
# Safe to re-run: any model whose .h5 already exists is loaded instead of
# retrained, and any experiment whose results.csv already exists is skipped.

wall_start = time.time()

print('=' * 72)
print('RUNNING ALL 20 FFCA-PRUNED EXPERIMENTS')
print(f'  Keep decisions : {KEEP_DECISIONS}')
print(f'  Ensemble size  : {N_ENSEMBLE} models per experiment')
print(f'  Max epochs     : {MAX_EPOCHS} (early stopping patience={PATIENCE})')
print(f'  Val year       : {VAL_YEAR}  |  Test year: 2024')
print(f'  Output root    : {RESULTS_ROOT}/')
print('=' * 72)

all_results = []
failed      = []

for idx, cfg in enumerate(EXPERIMENTS, start=1):
    print(f'\n[{idx:02d}/20] {cfg["name"]}')
    print(f'  lead={cfg["lead_time"]}h | '
          f'layers={cfg["hp"]["num_layers"]} | '
          f'neurons={cfg["hp"]["neurons"]} | '
          f'lr={cfg["hp"]["lr"]}')
    try:
        df_res = run_experiment(cfg)
        all_results.append(df_res)
    except Exception as exc:
        import traceback
        print(f'  ERROR: {exc}')
        traceback.print_exc()
        failed.append(cfg['name'])
        # Continue with remaining experiments even if one fails

# ── Aggregate summary ─────────────────────────────────────────────────────────
if all_results:
    df_summary = pd.concat(all_results, ignore_index=True)
    summary_path = os.path.join(RESULTS_ROOT, 'ffca_pruned_summary.csv')
    df_summary.to_csv(summary_path, index=False)

    wall_elapsed = (time.time() - wall_start) / 60
    print('\n' + '=' * 72)
    print(f'ALL DONE  ({wall_elapsed:.1f} min total)')
    if failed:
        print(f'  FAILED experiments: {failed}')
    print(f'  Summary saved to: {summary_path}')
    print('=' * 72 + '\n')

    cols = ['experiment', 'n_features', 'CF_15CM', 'CF_5CM', 'CF_1CM', 'RMSE', 'R2']
    pd.set_option('display.float_format', '{:.4f}'.format)
    display(df_summary[cols].sort_values('experiment'))

RUNNING ALL 20 FFCA-PRUNED EXPERIMENTS
  Keep decisions : {'CONFIDENTLY KEEP', 'KEEP (stable)', 'MONITOR (borderline)'}
  Ensemble size  : 30 models per experiment
  Max epochs     : 10000 (early stopping patience=20)
  Val year       : 2023  |  Test year: 2024
  Output root    : results/

[01/20] 3hr_measured_sigmoid
  lead=3h | layers=1 | neurons=100 | lr=0.001
  [skip] 3hr_measured_sigmoid_ffca — results.csv already exists

[02/20] 6hr_measured_sigmoid
  lead=6h | layers=1 | neurons=200 | lr=0.001
  [skip] 6hr_measured_sigmoid_ffca — results.csv already exists

[03/20] 12hr_measured_sigmoid
  lead=12h | layers=1 | neurons=100 | lr=0.001
  [skip] 12hr_measured_sigmoid_ffca — results.csv already exists

[04/20] 24hr_measured_sigmoid
  lead=24h | layers=1 | neurons=200 | lr=0.001
  [skip] 24hr_measured_sigmoid_ffca — results.csv already exists

[05/20] 3hr_perfect_prog_wls_sigmoid
  lead=3h | layers=1 | neurons=200 | lr=0.001
  [skip] 3hr_perfect_prog_wls_sigmoid_ffca — results.csv alr

  Done in 16.0 min | RMSE=1.93 cm  R²=0.973  CF(1cm)=74.2%
  Stop epochs (trained): [129, 51, 119, 95, 90, 79, 64, 108, 91, 106, 82]

[19/20] 12hr_perfect_prog_all_inputs_sigmoid
  lead=12h | layers=1 | neurons=200 | lr=0.001
  Selected 96 features: {'gwl': '-24..0 (17)', 'wl': '-19..3 (16)', 'rain': '-24..12 (37)', 'stgH': '-24..0 (12)', 'stgT': '-24..-11 (2)', 'gate1': '0..11 (8)', 'gate2': '-2..4 (4)'}
  Shapes — train: (45526, 96), val: (8730, 96), test: (7105, 96)
  y bounds: [-0.0579, 1.3131] (buffered max)


  Done in 49.4 min | RMSE=2.20 cm  R²=0.964  CF(1cm)=71.2%
  Stop epochs (trained): [87, 52, 98, 106, 87, 75, 107, 131, 128, 122, 90, 85, 119, 109, 110, 120, 140, 115, 84, 78, 77, 103, 95, 125, 87, 112, 64, 92, 72, 105]

[20/20] 24hr_perfect_prog_all_inputs_sigmoid
  lead=24h | layers=1 | neurons=200 | lr=0.001
  Selected 95 features: {'gwl': '-14..0 (15)', 'wl': '-5..7 (7)', 'rain': '-23..24 (48)', 'stgH': '-14..0 (10)', 'gate1': '-3..21 (9)', 'gate2': '-22..0 (6)'}
  Shapes — train: (45539, 95), val: (8736, 95), test: (7093, 95)
  y bounds: [-0.0579, 1.3131] (buffered max)


  Done in 41.6 min | RMSE=2.96 cm  R²=0.936  CF(1cm)=60.8%
  Stop epochs (trained): [94, 111, 104, 111, 65, 40, 44, 126, 55, 72, 54, 120, 103, 33, 126, 123, 48, 91, 164, 31, 122, 78, 54, 37, 31, 31, 94, 92, 47, 115]

ALL DONE  (107.0 min total)
  Summary saved to: results\ffca_pruned_summary.csv



,experiment,n_features,CF_15CM,CF_5CM,CF_1CM,RMSE,R2
2,12hr_measured_sigmoid_ffca,51,99.4370,94.7080,65.5735,0.0367,0.9012
18,12hr_perfect_prog_all_inputs_sigmoid_ffca,96,99.7889,95.9465,71.2456,0.0220,0.9643
14,12hr_perfect_prog_gate_sigmoid_ffca,72,99.3948,95.1020,66.3476,0.0352,0.9089
10,12hr_perfect_prog_rain_sigmoid_ffca,74,99.7185,94.6798,67.6144,0.0254,0.9526
6,12hr_perfect_prog_wls_sigmoid_ffca,52,99.4229,94.6376,66.2913,0.0368,0.9005
3,24hr_measured_sigmoid_ffca,58,98.8862,89.6800,52.4743,0.0501,0.8160
19,24hr_perfect_prog_all_inputs_sigmoid_ffca,95,99.6193,91.8229,60.7641,0.0296,0.9359
15,24hr_perfect_prog_gate_sigmoid_ffca,62,98.6888,89.8350,52.3615,0.0422,0.8696
11,24hr_perfect_prog_rain_sigmoid_ffca,74,99.4220,90.2721,55.1389,0.0346,0.9124
7,24hr_perfect_prog_wls_sigmoid_ffca,50,98.8439,89.6377,52.2628,0.0507,0.8114


In [7]:
# ─── Cell 6 (optional): Side-by-side comparison vs. original results ─────────
# Loads original results.csv files and compares with the FFCA-pruned results.

rows = []
for cfg in EXPERIMENTS:
    orig_path = os.path.join(RESULTS_ROOT, cfg['name'], 'MLP', 'test', 'results.csv')
    ffca_path = os.path.join(RESULTS_ROOT, cfg['name'] + '_ffca', 'MLP', 'test', 'results.csv')

    base = dict(experiment=cfg['name'], lead_time=cfg['lead_time'])

    for label, path in [('original', orig_path), ('ffca', ffca_path)]:
        if os.path.exists(path):
            r = pd.read_csv(path, index_col=0).iloc[0]
            rows.append({**base, 'variant': label,
                         'n_features': int(r.get('n_features', -1)),
                         'CF_15CM': r['CF_15CM'], 'CF_5CM': r['CF_5CM'],
                         'CF_1CM':  r['CF_1CM'],
                         'RMSE_cm': r['RMSE'] * 100,
                         'R2':      r['R2']})
        else:
            rows.append({**base, 'variant': label,
                         'n_features': None, 'CF_15CM': None, 'CF_5CM': None,
                         'CF_1CM': None, 'RMSE_cm': None, 'R2': None})

df_compare = pd.DataFrame(rows)

# Pivot to show original vs ffca side by side
df_pivot = df_compare.pivot_table(
    index=['experiment', 'lead_time'],
    columns='variant',
    values=['n_features', 'CF_1CM', 'RMSE_cm', 'R2'],
)
df_pivot.columns = ['_'.join(c) for c in df_pivot.columns]
df_pivot = df_pivot.reset_index().sort_values(['lead_time', 'experiment'])

# Compute RMSE and CF(1cm) delta (ffca - original; negative RMSE delta = improvement)
if 'RMSE_cm_ffca' in df_pivot and 'RMSE_cm_original' in df_pivot:
    df_pivot['RMSE_delta'] = df_pivot['RMSE_cm_ffca'] - df_pivot['RMSE_cm_original']
    df_pivot['CF1_delta']  = df_pivot['CF_1CM_ffca']  - df_pivot['CF_1CM_original']
    df_pivot['feat_reduction'] = df_pivot['n_features_original'] - df_pivot['n_features_ffca']

comparison_path = os.path.join(RESULTS_ROOT, 'ffca_vs_original_comparison.csv')
df_pivot.to_csv(comparison_path, index=False)
print(f'Comparison saved to: {comparison_path}\n')

pd.set_option('display.float_format', '{:.3f}'.format)
display(df_pivot)

Comparison saved to: results\ffca_vs_original_comparison.csv



,experiment,lead_time,CF_1CM_ffca,CF_1CM_original,R2_ffca,R2_original,RMSE_cm_ffca,RMSE_cm_original,n_features_ffca,n_features_original,RMSE_delta,CF1_delta,feat_reduction
10,3hr_measured_sigmoid,3,80.531,79.337,0.979,0.977,1.692,1.756,66.000,-1.000,-0.064,1.195,-67.000
11,3hr_perfect_prog_all_inputs_sigmoid,3,81.754,82.303,0.986,0.984,1.366,1.454,55.000,-1.000,-0.088,-0.548,-56.000
12,3hr_perfect_prog_gate_sigmoid,3,82.570,81.276,0.983,0.981,1.522,1.622,54.000,-1.000,-0.100,1.293,-55.000
13,3hr_perfect_prog_rain_sigmoid,3,81.515,80.447,0.987,0.983,1.351,1.518,64.000,-1.000,-0.167,1.068,-65.000
14,3hr_perfect_prog_wls_sigmoid,3,81.248,79.632,0.981,0.978,1.624,1.737,57.000,-1.000,-0.113,1.617,-58.000
15,6hr_measured_sigmoid,6,71.607,72.367,0.950,0.950,2.606,2.613,60.000,-1.000,-0.007,-0.759,-61.000
16,6hr_perfect_prog_all_inputs_sigmoid,6,74.223,75.517,0.973,0.972,1.926,1.948,77.000,-1.000,-0.022,-1.294,-78.000
17,6hr_perfect_prog_gate_sigmoid,6,73.337,74.996,0.958,0.961,2.399,2.288,62.000,-1.000,0.111,-1.659,-63.000
18,6hr_perfect_prog_rain_sigmoid,6,72.676,73.618,0.969,0.966,2.044,2.151,66.000,-1.000,-0.107,-0.942,-67.000
19,6hr_perfect_prog_wls_sigmoid,6,72.029,72.423,0.948,0.949,2.668,2.639,44.000,-1.000,0.029,-0.394,-45.000
